In [2]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 5.4 MB/s eta 0:00:00


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pypiahmad/goodreads-book-reviews1")

100%|██████████| 8.14G/8.14G [01:50<00:00, 78.7MB/s]

Extracting files...


In [4]:
file_path = path + "/goodreads_reviews_dedup.json"

In [5]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [35]:
# ================================
# 1. IMPORTS
# ================================
import pandas as pd
import numpy as np
import re
import emoji
import nltk
import joblib

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# ================================
# 2. DOWNLOAD NLTK DATA
# ================================
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# ================================
# 3. LOAD DATASET
# ================================
df = pd.read_json(file_path, lines=True, nrows=50000)
df = df[['review_text', 'rating']].dropna()

# ================================
# 4. LABEL CREATION
# ================================
def convert_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['rating'].apply(convert_sentiment)

# ================================
# 5. EMOJI ENRICHMENT (APPROACH 1)
# ================================
def enrich_with_emoji(text):
    em = emoji.demojize(text)
    em = em.replace(":", " ")
    return text + " " + em + " " + em   # 👈 double boost

def normalize_contractions(text):
    text = re.sub(r"n\'t", " not", text)
    text = re.sub(r"\'re", " are", text)
    text = re.sub(r"\'s", " is", text)
    text = re.sub(r"\'ll", " will", text)
    text = re.sub(r"\'ve", " have", text)
    text = re.sub(r"\'m", " am", text)
    return text

# ================================
# 6. PREPROCESSING
# ================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = normalize_contractions(text)
    text = enrich_with_emoji(text)  # integrate emoji meaning
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s!?]', '', text)

    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)

df['clean_review'] = df['review_text'].apply(preprocess_text)

# ================================
# 7. TF-IDF + SVM MODEL
# ================================
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,3),   # ↑ capture phrases like "not good at all"
    min_df=2,            # ↑ keep more useful rare terms
    max_df=0.9,
    sublinear_tf=True
)

X = vectorizer.fit_transform(df['clean_review'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LinearSVC(class_weight='balanced')
model.fit(X_train, y_train)

# ================================
# 8. EVALUATION
# ================================
y_pred = model.predict(X_test)



# ================================
# 9. CONTEXT-AWARE HEURISTIC (APPROACH 4)
# ================================
positive_emojis = ["smiling", "heart", "love", "laugh"]
negative_emojis = ["angry", "sad", "cry", "disappointed"]

def context_override(text, prediction):
    text_lower = text.lower()

    # strong negation patterns
    if "not good" in text_lower or "not great" in text_lower:
        return "negative"

    # BUT rule (dominant clause after 'but')
    if "but" in text_lower:
        parts = text_lower.split("but")
        if len(parts) > 1:
            return model.predict(
                vectorizer.transform([preprocess_text(parts[-1])])
            )[0]

    return prediction

# ================================
# 10. FINAL PREDICTION FUNCTION
# ================================
def predict_sentiment(text):
    if not text.strip():
        return "neutral"

    cleaned = preprocess_text(text)
    vec = vectorizer.transform([cleaned])

    if vec.nnz == 0:
        return "neutral"

    pred = model.predict(vec)[0]

    # apply context-aware correction
    final_pred = context_override(text, pred)

    return final_pred
# ================================
# 12. SAVE MODEL
# ================================


joblib.dump({
    "model": model,
    "vectorizer": vectorizer
}, "sentiment_model.pkl")

print("Model saved successfully!")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Model saved successfully!


In [36]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6888
              precision    recall  f1-score   support

    negative       0.52      0.50      0.51      1468
     neutral       0.45      0.42      0.43      2210
    positive       0.80      0.83      0.81      6322

    accuracy                           0.69     10000
   macro avg       0.59      0.58      0.59     10000
weighted avg       0.68      0.69      0.68     10000



In [37]:
# Rating Count
print("Rating Count:")
print(df['rating'].value_counts().sort_index())

# Rating Percentage
print("\nRating Percentage:")
print((df['rating'].value_counts(normalize=True) * 100).sort_index())



Rating Count:
rating
0     1966
1     1433
2     3941
3    11052
4    17636
5    13972
Name: count, dtype: int64

Rating Percentage:
rating
0     3.932
1     2.866
2     7.882
3    22.104
4    35.272
5    27.944
Name: proportion, dtype: float64


In [38]:
# ================================
# LOAD MODEL
# ================================
import joblib
import re
import emoji
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

data = joblib.load("sentiment_model.pkl")

model = data["model"]
vectorizer = data["vectorizer"]

print("Model loaded successfully!")


# ================================
# PREPROCESSING (SAME AS TRAINING)
# ================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def enrich_with_emoji(text):
    em = emoji.demojize(text)
    em = em.replace(":", " ")
    return text + " " + em + " " + em

def preprocess_text(text):
    text = enrich_with_emoji(text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s!?]', '', text)

    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)


# ================================
# CONTEXT RULE (SAME)
# ================================
def context_override(text, prediction):
    text_lower = text.lower()

    # strong negation patterns
    if "not good" in text_lower or "not great" in text_lower:
        return "negative"

    # BUT rule (dominant clause after 'but')
    if "but" in text_lower:
        parts = text_lower.split("but")
        if len(parts) > 1:
            return model.predict(
                vectorizer.transform([preprocess_text(parts[-1])])
            )[0]

    return prediction


# ================================
# FINAL PREDICTION FUNCTION
# ================================
def predict_sentiment(text):
    if not text.strip():
        return "neutral"

    cleaned = preprocess_text(text)
    vec = vectorizer.transform([cleaned])

    if vec.nnz == 0:
        return "neutral"

    pred = model.predict(vec)[0]
    return context_override(text, pred)


# ================================
# TEST EXAMPLES
# ================================
print(f"This product is amazing → {predict_sentiment('This product is amazing')}")
print(f"I love this 😍 → {predict_sentiment('I love this 😍')}")
print(f"I hate this 😡 → {predict_sentiment('I hate this 😡')}")
print(f"Not good 😍 → {predict_sentiment('Not good 😍')}")
print(f"This is sad 😢 but well written → {predict_sentiment('This is sad 😢 but well written')}")
print(f"It’s decent for the price → {predict_sentiment('It’s decent for the price')}")
print(f"The story was alright, but it didn’t leave a strong impression → {predict_sentiment('The story was alright, but it didn’t leave a strong impression')}")
print(f"It had a few good moments, but overall it was average → {predict_sentiment('It had a few good moments, but overall it was average')}")
print(f"The product works fine, but I expected a bit more → {predict_sentiment('The product works fine, but I expected a bit more')}")
print(f"The writing style is decent, though the pacing could have been better → {predict_sentiment('The writing style is decent, though the pacing could have been better')}")
print(f"The book has an interesting concept, but the execution feels uneven in places → {predict_sentiment('The book has an interesting concept, but the execution feels uneven in places')}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Model loaded successfully!
This product is amazing → positive
I love this 😍 → positive
I hate this 😡 → negative
Not good 😍 → negative
This is sad 😢 but well written → positive
It’s decent for the price → neutral
The story was alright, but it didn’t leave a strong impression → positive
It had a few good moments, but overall it was average → neutral
The product works fine, but I expected a bit more → neutral
The writing style is decent, though the pacing could have been better → neutral
The book has an interesting concept, but the execution feels uneven in places → neutral
